# Momentum — Execution Scenario Analysis
Tests how portfolio OOS performance changes depending on when trades are filled.
Hourly data is fetched **once** in the first cell — swap execution time in the cell below without re-fetching.

| Scenario | Description |
|---|---|
| `close` | Baseline — fill at prior day close |
| `Xh` | Fill at X:00 UTC on execution day (e.g. `10` = 10am UTC) |
| `worst_long` | True worst case for longs — enter at day High, exit at day Low |

In [ ]:
import os
import sys
import pandas as pd

# ── repo root — works on both Mac and Windows ─────────────────────────────────
ROOT   = os.path.expanduser('~/Desktop/epsilon/github/Epsilon-Quant-Research')
# ROOT = r'C:\Users\user\Documents\Epsilon Fund\Epsilon-Quant-Research'  # ← Windows path

WF_DIR = os.path.join(ROOT, 'topics', 'momentum', 'strategies', 'wf_testing')

sys.path.append(os.path.join(ROOT, 'infrastructure', 'data'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'walkforward'))
sys.path.append(os.path.join(ROOT, 'infrastructure', 'backtester'))

from binance_client import get_binance_client
from wf_visualizer import plot_portfolio_oos

client = get_binance_client()

---
## 1 — Load OOS results and fetch hourly data (run once)

In [ ]:
# ── load daily OOS pkl files ───────────────────────────────────────────────────
coin_dfs = {}
for fname in os.listdir(WF_DIR):
    if fname.endswith('_oos.pkl'):
        coin = fname.replace('_oos.pkl', '').upper()
        coin_dfs[coin] = pd.read_pickle(os.path.join(WF_DIR, fname))

print(f'Loaded daily OOS: {list(coin_dfs.keys())}')

# ── fetch 1h data for every coin (one-time pull) ───────────────────────────────
hourly = {}   # coin → 1h DataFrame

for coin, df in coin_dfs.items():
    symbol = coin + 'USDT'
    start  = str(df.index[0].date())
    end    = str(df.index[-1].date())
    klines = client.get_historical_klines(symbol, '1h', start, end)

    h = pd.DataFrame(klines, columns=[
        'Time','Open','High','Low','Close','Volume',
        'Close_time','Quote_volume','Trades','Taker_base','Taker_quote','Ignore'
    ])
    h['Time'] = pd.to_datetime(h['Time'], unit='ms', utc=True)
    for col in ['Open','High','Low','Close']:
        h[col] = h[col].astype(float)
    h = h.set_index('Time')
    hourly[coin] = h
    print(f'  {coin}: {len(h)} hourly bars ({start} → {end})')

print('\nHourly data ready.')

---
## 2 — Set execution scenario (re-run this cell to switch, no re-fetch needed)

**Options:**
- `SCENARIO = 'close'` — baseline, prior day close
- `SCENARIO = 10` — fill at 10am UTC (any integer hour 0–23)
- `SCENARIO = 'worst_long'` — true worst case: enter at High, exit at Low

In [ ]:
SCENARIO = 10   # ← 'close' | integer hour (0-23) | 'worst_long'


def apply_scenario(coin_dfs, hourly, scenario):
    """
    Returns adjusted coin_dfs with Close replaced by execution-price series.
    position column is never modified.

    scenario = 'close'       → no change
    scenario = int (0-23)    → fill at that UTC hour each day
    scenario = 'worst_long'  → entries fill at High, exits fill at Low
    """
    if scenario == 'close':
        return coin_dfs

    adjusted = {}

    for coin, df in coin_dfs.items():
        d = df.copy()
        d.index = d.index.normalize()  # ensure date-only index
        h = hourly[coin]

        if isinstance(scenario, int):
            # ── hourly fill: price at HH:00 UTC each day ───────────────────
            prices = (
                h[h.index.hour == scenario]['Close']
                .resample('1D').last()
            )
            prices.index = prices.index.tz_localize(None).normalize()
            d['Close'] = prices.reindex(d.index).ffill()

        elif scenario == 'worst_long':
            # ── true worst case for longs ──────────────────────────────────
            # entry bar  (position goes from 0 → non-zero): fill at day High
            # exit bar   (position goes from non-zero → 0): fill at day Low
            # holding bars (no change): use close (P&L is path-independent)

            daily_high = h['High'].resample('1D').max()
            daily_low  = h['Low'].resample('1D').min()
            daily_high.index = daily_high.index.tz_localize(None).normalize()
            daily_low.index  = daily_low.index.tz_localize(None).normalize()

            pos      = d['position']
            prev_pos = pos.shift(1).fillna(0)

            is_entry = (prev_pos == 0) & (pos != 0)
            is_exit  = (prev_pos != 0) & (pos == 0)

            exec_price = d['Close'].copy()
            exec_price[is_entry] = daily_high.reindex(d.index).ffill()[is_entry]
            exec_price[is_exit]  = daily_low.reindex(d.index).ffill()[is_exit]

            d['Close'] = exec_price

        adjusted[coin] = d

    return adjusted


coin_dfs_exec = apply_scenario(coin_dfs, hourly, SCENARIO)
label = f'{SCENARIO}h UTC' if isinstance(SCENARIO, int) else SCENARIO
print(f'Scenario: {label}')

---
## 3 — Portfolio OOS performance

In [ ]:
SHOW_COINS = ['ETH', 'XRP', 'AVAX', 'SOL', 'BNB']   # or list(coin_dfs.keys())

metrics = plot_portfolio_oos(
    coin_dfs   = coin_dfs_exec,
    show_coins = SHOW_COINS,
    benchmark  = {'BTC': coin_dfs['BTC']},   # baseline always uses original close
    show       = True,
    save_html  = os.path.join(
        ROOT, 'topics', 'momentum', 'results',
        f'portfolio_{label}.html'
    ),
)

print(f'\n── Scenario: {label} ──')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<20} {v:.4f}')